# Suite PAR — Partial evaluation

`Partials.OfAny(expression, variables)` evaluates what the variables it is given determine and leaves the rest a Basic
expression, the residual: a known subexpression becomes the literal of its value, Kleene's logic decides `and`, `or`
and `implies` where it can, and a let of a known value disappears once its body no longer refers to it. The residual
agrees with the expression wherever the expression evaluates without raising.

In [ ]:
import { Domains as D, Evaluators as V, Expressions as E, Partials as PA } from "@mbse/expressions";
import * as Python from "@mbse/expressions/Dialects/Python";
import { Terms as F } from "@mbse/expressions/Framework";
import * as T from "@mbse/expressions/Translators";
import { Errors, JSON, Proxies, Schemas as S } from "@mbse/schemas/Framework";

const { ValueError } = Errors;
const P = Python.Expressions;
import { assert, raises, text } from "./support.js";

const Person = new S.OfObject.Builder().ref().properties(text("name"), text("age", BigInt), text("email")).create();
Proxies.register("Person", Person);
const PB = (Proxies.Builders as any).Person;
const ann = PB().name("Ann").age(30n).email("ann@example.com").create();
const bob = PB().name("Bob").age(70n).create(); // no email
const [self, x, y, xs] = [E.variable("this"), E.variable("x"), E.variable("y"), E.variable("xs")];
const I8 = new D.OfInteger.Data(8n);

/** Whether a residual is the expected expression. */
function same(residual: unknown, expected: unknown): boolean {
  return F.same(residual, E.OfAny.resolve(expected as never));
}

## PAR-01 · Known variables are evaluated and unknown ones stay; a known subexpression becomes the literal of its value

In [ ]:
{
  assert(same(PA.OfAny(x.add(E.literal(1n).add(2n))), x.add(3n)) && same(PA.OfAny(x.add(1n), { x: 2n }), 3n));
  assert(same(PA.OfAny(E.literal(1n).add(2n).add(x)), E.literal(3n).add(x)) && same(PA.OfAny(x), x));
  assert(same(PA.OfAny(x.add(E.literal(1n, I8)), { x: new D.Value(I8, 2n) }), E.literal(3n, I8))); // a typed literal
  assert(same(PA.OfAny(self.age.ge(18n), { this: ann }), true) && same(PA.OfAny(self.has("email"), { this: bob }), false));
  // A value without a literal keeps its subexpression, which still refers to what it needs.
  assert(same(PA.OfAny(self, { this: ann }), self) && same(PA.OfAny(xs, { xs: [1n, 2n] }), xs));
  assert(same(PA.OfAny(xs.count(), { xs: [1n, 2n] }), 2n) && same(PA.OfAny(xs.count().add(y), { xs: [1n, 2n] }), E.literal(2n).add(y)));
  const unknown = self.get("phone").ge(18n);
  assert(same(PA.OfAny(unknown, { this: ann }), unknown) && same(PA.OfAny(unknown.or_(y), { this: ann }), unknown.or_(y)));
  assert(same(PA.OfAny((b: any) => b.as_literal(1n)), 1n) && same(PA.OfAny(1n), 1n)); // any Spec
}

## PAR-02 · Kleene's logic decides and, or and implies with what is known, and drops a neutral operand

In [ ]:
{
  const rule = x.gt(1n).and_(y.lt(2n));
  assert(same(PA.OfAny(rule, { x: 5n }), y.lt(2n)) && same(PA.OfAny(rule, { x: 0n }), false));
  assert(same(PA.OfAny(rule, { y: 1n }), x.gt(1n)) && same(PA.OfAny(rule, { y: 5n }), false) && same(PA.OfAny(rule), rule));
  const either = x.gt(1n).or_(y.lt(2n));
  assert(same(PA.OfAny(either, { x: 5n }), true) && same(PA.OfAny(either, { x: 0n }), y.lt(2n)));
  assert(same(PA.OfAny(either, { y: 1n }), true) && same(PA.OfAny(either, { y: 5n }), x.gt(1n)));
  const then = x.implies(y);
  assert(same(PA.OfAny(then, { x: false }), true) && same(PA.OfAny(then, { y: true }), true));
  assert(same(PA.OfAny(then, { x: true }), y) && same(PA.OfAny(then, { y: false }), x.implies(false)) && same(PA.OfAny(then), then));
  // An unknown value decides nothing: the operand stays.
  assert(same(PA.OfAny(rule, { x: null }), rule) && same(PA.OfAny(x.not_(), {}), x.not_()));
  assert(same(PA.OfAny(E.operation("xor", x, true)), E.operation("xor", x, true))); // an extension is only rebuilt
}

## PAR-03 · A let of a known value binds it in its body and stays while the body refers to it; names hide known variables

In [ ]:
{
  const a = E.variable("a");
  const rule = E.let_("a", x.add(1n), a.mul(y));
  assert(same(PA.OfAny(rule, { x: 2n }), E.literal(3n).mul(y)) && same(PA.OfAny(rule, { y: 2n }), E.let_("a", x.add(1n), a.mul(2n))));
  assert(same(PA.OfAny(rule), rule) && same(PA.OfAny(rule, { x: 2n, y: 2n }), 6n));
  const o = E.variable("o");
  assert(same(PA.OfAny(E.let_("o", self, o.age.add(y)), { this: ann }), E.literal(30n).add(y))); // dropped: no longer needed
  assert(same(PA.OfAny(E.let_("o", self, o.add(y)), { this: ann }), E.let_("o", self, o.add(y)))); // an object: kept
  assert(same(PA.OfAny(E.let_("x", y, x.add(1n)), { x: 5n }), E.let_("x", y, x.add(1n)))); // the let hides the known x
  // Quantifiers: evaluated when known; otherwise their collection and body are reduced, the name hiding a known variable.
  const v = E.variable("v");
  assert(same(PA.OfAny(xs.all("v", v.gt(y)), { xs: [1n, 2n], y: 0n }), true));
  assert(same(PA.OfAny(xs.all("v", v.gt(y)), { y: 0n }), xs.all("v", v.gt(0n))));
  assert(same(PA.OfAny(xs.all("v", v.gt(y)), { xs: [1n, 2n] }), xs.all("v", v.gt(y))));
  assert(same(PA.OfAny(xs.all("y", y.gt(0n)), { y: 5n }), xs.all("y", y.gt(0n))));
}

## PAR-04 · The residual agrees with evaluation in every scope that completes the known one, and stores and translates

In [ ]:
{
  const rule = self.age.ge(18n).and_(x.lt(65n).or_(self.has("email").not_())).implies(y.or_(x.eq(3n)));
  const values: Record<string, unknown[]> = { this: [ann, bob], x: [3n, 70n], y: [true, false, null] };
  const names = Object.keys(values);
  const product = (keys: string[]): Record<string, unknown>[] => keys.reduce<Record<string, unknown>[]>(
    (scopes, key) => scopes.flatMap((scope) => (values[key] as unknown[]).map((value) => ({ ...scope, [key]: value }))), [{}]);
  for (let mask = 0; mask < 1 << names.length; mask++) {
    const given = names.filter((_, i) => mask & (1 << i));
    for (const known of product(given)) {
      const residual = PA.OfAny(rule, known);
      for (const rest of product(names.filter((name) => !given.includes(name)))) {
        const scope = { ...known, ...rest };
        assert(V.OfAny(residual as never, scope) === V.OfAny(rule, scope), globalThis.JSON.stringify(Object.keys(known)));
      }
    }
  }
  const residual = PA.OfAny(rule, { this: bob });
  assert(same(residual, y.or_(x.eq(3n)))); // Bob is an adult with no email: the condition holds
  const back = JSON.FromJSON(E.Builders).Reachable(E.OfOperation.Schema, JSON.ToJSON.Reachable(E.OfOperation.Schema, residual as never));
  assert(F.same(back, residual) && P.render(T.between(E.DIALECT, P.DIALECT).forward(residual)) === "y or x == 3");
  // Evaluating a known subexpression throws as evaluation would; so does a cycle.
  raises(TypeError, () => PA.OfAny(x.add(E.literal(1n).add("a"))), "add expects numbers of one domain, got int and str");
  const cyclic = E.operation("add", 1n, x).data as any;
  cyclic.arguments = [cyclic, x.data];
  raises(ValueError, () => PA.OfAny(cyclic), "the expression contains a cycle");
  raises(TypeError, () => PA.REDUCER.run("x"), "not an expression: 'x'");
}